# VL-JEPA: 2-GPU Training

**Multi-GPU setup:**
- Vision encoder (frozen) on GPU 0
- Predictor + Text encoder on both GPUs (DataParallel)
- Automatically splits batches across GPUs

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, AutoVideoProcessor, LlamaForCausalLM
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm
import numpy as np

# Check GPUs
assert torch.cuda.device_count() >= 2, f'Need 2 GPUs, found {torch.cuda.device_count()}'
print(f'Found {torch.cuda.device_count()} GPUs')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory/1e9:.1f}GB)')

device0 = torch.device('cuda:0')
device1 = torch.device('cuda:1')

## 1. Load V-JEPA2 Vision Encoder (GPU 0)

In [ ]:
print('Loading V-JEPA2 on GPU 0 (frozen)...')

# Load V-JEPA2 model and processor
model_id = 'facebook/vjepa2-vitl-fpc64-256'
print(f'Loading {model_id}...')

vision_processor = AutoVideoProcessor.from_pretrained(model_id)
vision_model = AutoModel.from_pretrained(
    model_id,
    torch_dtype=torch.float16
).to(device0)

print(f'✓ Loaded V-JEPA2: {model_id}')

for param in vision_model.parameters():
    param.requires_grad = False
vision_model.eval()

vision_dim = vision_model.config.hidden_size
print(f'Vision encoder: {sum(p.numel() for p in vision_model.parameters())/1e6:.1f}M params on GPU 0')

## 2. Load Llama Predictor (Both GPUs)

In [ ]:
print('\nLoading Llama-3.2-1B...')
llama = LlamaForCausalLM.from_pretrained(
    'meta-llama/Llama-3.2-1B',
    torch_dtype=torch.float16
)

# Extract layers (will be moved to DataParallel later)
predictor_layers = nn.ModuleList(llama.model.layers[8:16])
predictor_dim = llama.config.hidden_size

print(f'✓ Extracted layers 8-15')
print(f'Predictor: {sum(p.numel() for p in predictor_layers.parameters())/1e6:.1f}M params')

# Input projection (will be on DataParallel)
predictor_proj_in = nn.Linear(vision_dim, predictor_dim)

## 3. Load Gemma Text Encoder (Both GPUs)

In [ ]:
print('\nLoading Gemma-2-2B...')
text_model = AutoModel.from_pretrained(
    'google/gemma-2-2b',
    torch_dtype=torch.float16
)
text_tokenizer = AutoTokenizer.from_pretrained('google/gemma-2-2b')

if text_tokenizer.pad_token is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token

text_dim = text_model.config.hidden_size
print(f'✓ Loaded Gemma')
print(f'Text encoder: {sum(p.numel() for p in text_model.parameters())/1e6:.1f}M params')

## 4. Load Dataset

In [ ]:
class VLJEPACore(nn.Module):
    """Trainable components (will be wrapped in DataParallel)."""
    def __init__(self, pred_layers, proj_in, text, pred_dim, text_dim, embed_dim=2048):
        super().__init__()
        self.pred_layers = pred_layers
        self.proj_in = proj_in
        self.text = text
        self.pred_proj = nn.Linear(pred_dim, embed_dim)
        self.text_proj = nn.Linear(text_dim, embed_dim)
        # IMPORTANT: Initialize temperature as float16
        self.temperature = nn.Parameter(torch.tensor(0.07, dtype=torch.float16))
    
    def forward(self, visual, input_ids, attention_mask):
        # Predictor path
        h = self.proj_in(visual)
        for layer in self.pred_layers:
            h = layer(h, use_cache=False)[0]
        pred = F.normalize(self.pred_proj(h), dim=-1)
        
        # Text encoder path
        text_out = self.text(input_ids=input_ids, attention_mask=attention_mask)
        target = F.normalize(self.text_proj(text_out.last_hidden_state), dim=-1)
        
        return pred, target

class VLJEPA2GPU(nn.Module):
    """Full model with frozen vision encoder on GPU 0."""
    def __init__(self, vision_model, core_model):
        super().__init__()
        self.vision = vision_model
        self.core = core_model
    
    def forward(self, pixels, input_ids, attention_mask):
        B, T, C, H, W = pixels.shape
        
        # Vision encoding on GPU 0
        with torch.no_grad():
            pixels_gpu0 = pixels.to(device0, dtype=torch.float16)
            v_out = self.vision(pixel_values_videos=pixels_gpu0.view(B*T, C, H, W))
            if hasattr(v_out, 'pooler_output') and v_out.pooler_output is not None:
                visual = v_out.pooler_output
            else:
                visual = v_out.last_hidden_state[:, 0]
        visual = visual.view(B, T, -1)
        
        # DataParallel automatically handles moving to GPU 1 and splitting
        pred, target = self.core(visual, input_ids, attention_mask)
        return pred, target
    
    def compute_loss(self, pred, target, mask):
        pred_pool = pred.mean(dim=1)
        target_pool = (target * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp(min=1)
        pred_pool = F.normalize(pred_pool, dim=-1)
        target_pool = F.normalize(target_pool, dim=-1)
        
        sim = torch.matmul(pred_pool, target_pool.T) / self.core.module.temperature
        labels = torch.arange(sim.shape[0], device=sim.device)
        loss_v2t = F.cross_entropy(sim, labels)
        loss_t2v = F.cross_entropy(sim.T, labels)
        return (loss_v2t + loss_t2v) / 2

# Build core (trainable parts)
core = VLJEPACore(predictor_layers, predictor_proj_in, text_model, predictor_dim, text_dim)

# Convert all parameters to float16
core = core.half()

# Wrap core in DataParallel
core_parallel = nn.DataParallel(core, device_ids=[0, 1])
core_parallel.to(device0)

# Build full model
model = VLJEPA2GPU(vision_model, core_parallel)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\\nTotal: {total/1e6:.1f}M | Trainable: {trainable/1e6:.1f}M | Frozen: {(total-trainable)/1e6:.1f}M')
print(f'Trainable parts distributed across GPU 0 and GPU 1')
print(f'\\nDtype check:')
print(f'  Predictor: {next(core_parallel.module.pred_layers.parameters()).dtype}')
print(f'  Text model: {next(core_parallel.module.text.parameters()).dtype}')
print(f'  Temperature: {core_parallel.module.temperature.dtype}')

## 5. Dataset Class

In [ ]:
class VideoTextDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, processor, num_frames=16):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.processor = processor
        self.num_frames = num_frames
        self.has_image = 'image' in hf_dataset[0]
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        sample = self.dataset[idx]
        
        if self.has_image:
            img = sample['image'].convert('RGB')
            frames = [img] * self.num_frames
        else:
            frames = [Image.new('RGB', (224, 224))] * self.num_frames
        
        # Video processor expects 'videos' parameter
        pixels = self.processor(videos=frames, return_tensors='pt')['pixel_values']
        
        if 'caption' in sample:
            caption = sample['caption']
        elif 'sentences' in sample:
            caption = sample['sentences']['raw'][0]
        else:
            caption = 'A video.'
        
        tokens = self.tokenizer(
            caption,
            return_tensors='pt',
            padding='max_length',
            max_length=77,
            truncation=True
        )
        
        return {
            'pixels': pixels,
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
        }

train_dataset = VideoTextDataset(dataset, text_tokenizer, vision_processor)
print(f'✓ Dataset: {len(train_dataset)} samples')

## 6. Multi-GPU Model

In [ ]:
class VLJEPACore(nn.Module):
    """Trainable components (will be wrapped in DataParallel)."""
    def __init__(self, pred_layers, proj_in, text, pred_dim, text_dim, embed_dim=2048):
        super().__init__()
        self.pred_layers = pred_layers
        self.proj_in = proj_in
        self.text = text
        self.pred_proj = nn.Linear(pred_dim, embed_dim)
        self.text_proj = nn.Linear(text_dim, embed_dim)
        self.temperature = nn.Parameter(torch.tensor(0.07))
    
    def forward(self, visual, input_ids, attention_mask):
        # Predictor path
        h = self.proj_in(visual)
        for layer in self.pred_layers:
            h = layer(h, use_cache=False)[0]
        pred = F.normalize(self.pred_proj(h), dim=-1)
        
        # Text encoder path
        text_out = self.text(input_ids=input_ids, attention_mask=attention_mask)
        target = F.normalize(self.text_proj(text_out.last_hidden_state), dim=-1)
        
        return pred, target

class VLJEPA2GPU(nn.Module):
    """Full model with frozen vision encoder on GPU 0."""
    def __init__(self, vision_model, core_model):
        super().__init__()
        self.vision = vision_model  # On GPU 0, frozen
        self.core = core_model      # DataParallel on both GPUs
    
    def forward(self, pixels, input_ids, attention_mask):
        B, T, C, H, W = pixels.shape
        
        # Vision encoding on GPU 0
        with torch.no_grad():
            pixels_gpu0 = pixels.to(device0)
            v_out = self.vision(pixel_values_videos=pixels_gpu0.view(B*T, C, H, W))
            if hasattr(v_out, 'pooler_output') and v_out.pooler_output is not None:
                visual = v_out.pooler_output
            else:
                visual = v_out.last_hidden_state[:, 0]
        visual = visual.view(B, T, -1)
        
        # DataParallel automatically handles moving to GPU 1 and splitting
        pred, target = self.core(visual, input_ids, attention_mask)
        return pred, target
    
    def compute_loss(self, pred, target, mask):
        pred_pool = pred.mean(dim=1)
        target_pool = (target * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp(min=1)
        pred_pool = F.normalize(pred_pool, dim=-1)
        target_pool = F.normalize(target_pool, dim=-1)
        
        sim = torch.matmul(pred_pool, target_pool.T) / self.core.module.temperature
        labels = torch.arange(sim.shape[0], device=sim.device)
        loss_v2t = F.cross_entropy(sim, labels)
        loss_t2v = F.cross_entropy(sim.T, labels)
        return (loss_v2t + loss_t2v) / 2

# Build core (trainable parts)
core = VLJEPACore(predictor_layers, predictor_proj_in, text_model, predictor_dim, text_dim)

# Wrap core in DataParallel for GPUs 0 and 1
core_parallel = nn.DataParallel(core, device_ids=[0, 1])
core_parallel.to(device0)

# Build full model
model = VLJEPA2GPU(vision_model, core_parallel)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal: {total/1e6:.1f}M | Trainable: {trainable/1e6:.1f}M | Frozen: {(total-trainable)/1e6:.1f}M')
print(f'Trainable parts distributed across GPU 0 and GPU 1')

## 7. Training Setup

In [ ]:
# DataLoader - larger batch size for 2 GPUs
batch_size = 8  # 4 per GPU
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)
print(f'DataLoader: {len(train_loader)} batches (batch_size={batch_size}, {batch_size//2} per GPU)\n')

# Optimizer
base_lr = 1e-4
text_lr = base_lr * 0.05

text_params = list(core_parallel.module.text.parameters()) + list(core_parallel.module.text_proj.parameters())
other_params = [p for p in core_parallel.parameters() if p.requires_grad and not any(p is tp for tp in text_params)]

optimizer = torch.optim.AdamW([
    {'params': other_params, 'lr': base_lr},
    {'params': text_params, 'lr': text_lr}
], weight_decay=0.01)

print(f'Base LR: {base_lr}')
print(f'Text LR: {text_lr}')

## 8. Training Loop

In [ ]:
max_steps = 500
log_interval = 50

print(f'\nTraining for {max_steps} steps on 2 GPUs...\n')

model.train()
core_parallel.train()
global_step = 0

for batch in tqdm(train_loader, total=max_steps):
    if global_step >= max_steps:
        break
    
    pixels = batch['pixels']  # Don't move yet, vision encoder will handle it
    input_ids = batch['input_ids'].to(device0)  # Start on GPU 0
    attention_mask = batch['attention_mask'].to(device0)
    
    # Forward (DataParallel automatically splits across GPUs)
    pred, target = model(pixels, input_ids, attention_mask)
    loss = model.compute_loss(pred, target, attention_mask)
    
    # Backward
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    global_step += 1
    
    if global_step % log_interval == 0:
        temp = core_parallel.module.temperature.item()
        print(f'Step {global_step} | Loss: {loss.item():.4f} | Temp: {temp:.4f}')

print('\n✓ Training complete!')

## 9. Save Checkpoint

In [ ]:
# Save only trainable parts (from DataParallel module)
checkpoint = {
    'predictor_layers': core_parallel.module.pred_layers.state_dict(),
    'proj_in': core_parallel.module.proj_in.state_dict(),
    'pred_proj': core_parallel.module.pred_proj.state_dict(),
    'text_model': core_parallel.module.text.state_dict(),
    'text_proj': core_parallel.module.text_proj.state_dict(),
    'temperature': core_parallel.module.temperature.data,
    'optimizer': optimizer.state_dict(),
}

torch.save(checkpoint, 'vljepa_2gpu_checkpoint.pt')
print('✓ Saved to vljepa_2gpu_checkpoint.pt')